Random Forest

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(240, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # random forest model
    model = RandomForestClassifier(

        # more trees
        n_estimators=700,

        # balanced tree depth
        max_depth=12,

        # reduce overfitting
        min_samples_split=4,
        min_samples_leaf=2,

        # better feature sampling
        max_features="sqrt",

        # handle imbalance
        class_weight="balanced",

        # bootstrap sampling
        bootstrap=True,

        random_state=42,
        n_jobs=-1
    )

    # train model
    model.fit(X_train, y_train)

    # predict probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # threshold
    THRESHOLD = 0.34

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 240)

Predicted Distribution:
[14 15]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.70      0.82        20
           1       0.60      1.00      0.75         9

    accuracy                           0.79        29
   macro avg       0.80      0.85      0.79        29
weighted avg       0.88      0.79      0.80        29

ROC-AUC Score: 0.8389

Confusion

Random Forest 5 folds

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # select important features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(250, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # random forest model
        model = RandomForestClassifier(

            # more trees
            n_estimators=700,

            # better generalization
            max_depth=12,

            # prevent overfitting
            min_samples_split=4,
            min_samples_leaf=2,

            # better feature sampling
            max_features="sqrt",

            # handle imbalance
            class_weight="balanced",

            # enable bootstrap
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 250)

----- Fold 1 -----
Fold ROC-AUC: 0.5722

----- Fold 2 -----
Fold ROC-AUC: 0.9188

----- Fold 3 -----
Fold ROC-AUC: 0.9187

----- Fold 4 -----
Fold ROC-AUC: 0.9

----- Fold 5 -----
Fold ROC-AUC: 0.8772

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.5722 0.9188 0.9188 0.9    0.8772]

Mean Fold ROC-AUC:
0.8374

Overall 

Random Forest 10 folds

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # select important features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(250, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # random forest model
        model = RandomForestClassifier(

            # more trees
            n_estimators=700,

            # better generalization
            max_depth=12,

            # prevent overfitting
            min_samples_split=4,
            min_samples_leaf=2,

            # better feature sampling
            max_features="sqrt",

            # handle imbalance
            class_weight="balanced",

            # enable bootstrap
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 250)

----- Fold 1 -----
Fold ROC-AUC: 0.38

----- Fold 2 -----
Fold ROC-AUC: 0.775

----- Fold 3 -----
Fold ROC-AUC: 0.85

----- Fold 4 -----
Fold ROC-AUC: 1.0

----- Fold 5 -----
Fold ROC-AUC: 0.925

----- Fold 6 -----
Fold ROC-AUC: 0.95

----- Fold 7 -----
Fold ROC-AUC: 0.975

----- Fold 8 -----
Fold ROC-AUC: 0.8

----- Fold 9 -----
Fold ROC-AUC: 0.975

----- Fold 10 -----
Fold ROC-AUC: 0.8444

--------------------------------------------------

XGBoost

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP MORE MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(400, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # HANDLE CLASS IMBALANCE
    # =====================================================
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 4))

    # =====================================================
    # TUNED MULTIMODAL XGBOOST
    # =====================================================
    model = XGBClassifier(

        # Core
        objective="binary:logistic",
        eval_metric="auc",

        # More boosting
        n_estimators=900,

        # Slightly deeper trees
        max_depth=7,

        # Slower learning
        learning_rate=0.015,

        # Weaker regularization
        reg_alpha=0.3,
        reg_lambda=2,

        # Better multimodal fusion
        subsample=0.9,
        colsample_bytree=0.9,

        # More flexible splits
        min_child_weight=1,
        gamma=0.2,

        # Handle imbalance
        scale_pos_weight=scale_pos_weight,

        # Faster and stable tree method
        tree_method="hist",

        random_state=42,
        n_jobs=-1
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # LOWER THRESHOLD
    # =====================================================
    THRESHOLD = 0.30

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 400)
Scale Pos Weight: 2.3939

Predicted Distribution:
[17 12]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.70      0.76        20
           1       0.50      0.67      0.57         9

    accuracy                           0.69        29
   macro avg       0.66      0.68      0.66        29
weighted avg       0.72      0.69      0.70        29

ROC-AUC Score: 0.6556

Confusion Matrix:
[[14  6]
 [ 3  6]]

TRAINING FOR PHASE 2
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 897)
Shape After SelectKBest: (141, 400)
Scale Pos Weight: 2.3939

Predicte

XGBoost 5 folds

In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(350, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # random forest model
        model = RandomForestClassifier(

            # more trees
            n_estimators=800,

            # better multimodal learning
            max_depth=14,

            # reduce overfitting
            min_samples_split=3,
            min_samples_leaf=1,

            # better feature selection
            max_features="sqrt",

            # handle imbalance
            class_weight="balanced",

            # bootstrap sampling
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.30

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 350)

----- Fold 1 -----
Fold ROC-AUC: 0.5222

----- Fold 2 -----
Fold ROC-AUC: 0.8812

----- Fold 3 -----
Fold ROC-AUC: 0.9375

----- Fold 4 -----
Fold ROC-AUC: 0.9688

----- Fold 5 -----
Fold ROC-AUC: 0.883

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.5222 0.8812 0.9375 0.9688 0.883 ]

Mean Fold ROC-AUC:
0.8386

Overal

XGBoost 10 folds

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(350, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # random forest model
        model = RandomForestClassifier(

            # more trees
            n_estimators=800,

            # better multimodal learning
            max_depth=14,

            # reduce overfitting
            min_samples_split=3,
            min_samples_leaf=1,

            # better feature selection
            max_features="sqrt",

            # handle imbalance
            class_weight="balanced",

            # bootstrap sampling
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.30

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 350)

----- Fold 1 -----
Fold ROC-AUC: 0.42

----- Fold 2 -----
Fold ROC-AUC: 0.725

----- Fold 3 -----
Fold ROC-AUC: 0.925

----- Fold 4 -----
Fold ROC-AUC: 0.975

----- Fold 5 -----
Fold ROC-AUC: 0.825

----- Fold 6 -----
Fold ROC-AUC: 0.9

----- Fold 7 -----
Fold ROC-AUC: 1.0

----- Fold 8 -----
Fold ROC-AUC: 0.875

----- Fold 9 -----
Fold ROC-AUC: 0.9

----- Fold 10 -----
Fold ROC-AUC: 0.8444

--------------------------------------------------

SVM

In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(240, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # feature scaling
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # balanced phase-1 svm
    model = SVC(

        # nonlinear boundary
        kernel="rbf",

        # balanced flexibility
        C=1.0,

        # moderate kernel smoothness
        gamma=0.004,

        # handle imbalance
        class_weight="balanced",

        # enable probabilities
        probability=True,

        random_state=42
    )

    # train model
    model.fit(X_train, y_train)

    # predict probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # threshold
    THRESHOLD = 0.34

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 240)

Predicted Distribution:
[19 10]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.75      0.77        20
           1       0.50      0.56      0.53         9

    accuracy                           0.69        29
   macro avg       0.64      0.65      0.65        29
weighted avg       0.70      0.69      0.69        29

ROC-AUC Score: 0.7833

Confusion

SVM 5 folds

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(240, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # balanced phase-1 svm
        model = SVC(

            # nonlinear boundary
            kernel="rbf",

            # balanced flexibility
            C=1.0,

            # moderate kernel smoothness
            gamma=0.004,

            # handle imbalance
            class_weight="balanced",

            # enable probabilities
            probability=True,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 240)

----- Fold 1 -----
Fold ROC-AUC: 0.6

----- Fold 2 -----
Fold ROC-AUC: 0.9125

----- Fold 3 -----
Fold ROC-AUC: 0.7875

----- Fold 4 -----
Fold ROC-AUC: 0.8813

----- Fold 5 -----
Fold ROC-AUC: 0.9035

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.6    0.9125 0.7875 0.8812 0.9035]

Mean Fold ROC-AUC:
0.817

Overall R

SVM 10 folds

In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(240, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # balanced phase-1 svm
        model = SVC(

            # nonlinear boundary
            kernel="rbf",

            # balanced flexibility
            C=1.0,

            # moderate kernel smoothness
            gamma=0.004,

            # handle imbalance
            class_weight="balanced",

            # enable probabilities
            probability=True,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 240)

----- Fold 1 -----
Fold ROC-AUC: 0.44

----- Fold 2 -----
Fold ROC-AUC: 0.725

----- Fold 3 -----
Fold ROC-AUC: 0.95

----- Fold 4 -----
Fold ROC-AUC: 0.9

----- Fold 5 -----
Fold ROC-AUC: 0.9

----- Fold 6 -----
Fold ROC-AUC: 0.725

----- Fold 7 -----
Fold ROC-AUC: 0.925

----- Fold 8 -----
Fold ROC-AUC: 0.875

----- Fold 9 -----
Fold ROC-AUC: 0.925

----- Fold 10 -----
Fold ROC-AUC: 0.9333

-------------------------------------------------

Logistic Regression

In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # feature scaling
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # logistic regression model
    model = LogisticRegression(

        # better convergence
        solver="saga",

        # moderate regularization
        C=0.8,

        # handle imbalance
        class_weight="balanced",

        # more iterations
        max_iter=5000,

        random_state=42
    )

    # train model
    model.fit(X_train, y_train)

    # predict probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # threshold
    THRESHOLD = 0.34

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

Predicted Distribution:
[17 12]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.75      0.81        20
           1       0.58      0.78      0.67         9

    accuracy                           0.76        29
   macro avg       0.73      0.76      0.74        29
weighted avg       0.79      0.76      0.77        29

ROC-AUC Score: 0.7833

Confusion

Logistic Regression 5 folds

In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # logistic regression model
        model = LogisticRegression(

            # better convergence
            solver="saga",

            # moderate regularization
            C=0.8,

            # handle imbalance
            class_weight="balanced",

            # more iterations
            max_iter=5000,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.7278

----- Fold 2 -----
Fold ROC-AUC: 0.825

----- Fold 3 -----
Fold ROC-AUC: 0.9187

----- Fold 4 -----
Fold ROC-AUC: 0.825

----- Fold 5 -----
Fold ROC-AUC: 0.848

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.7278 0.825  0.9188 0.825  0.848 ]

Mean Fold ROC-AUC:
0.8289

Overall 

Logistic Regression 10 folds

In [25]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # logistic regression model
        model = LogisticRegression(

            # better convergence
            solver="saga",

            # moderate regularization
            C=0.8,

            # handle imbalance
            class_weight="balanced",

            # more iterations
            max_iter=5000,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.6

----- Fold 2 -----
Fold ROC-AUC: 0.75

----- Fold 3 -----
Fold ROC-AUC: 0.825

----- Fold 4 -----
Fold ROC-AUC: 0.975

----- Fold 5 -----
Fold ROC-AUC: 1.0

----- Fold 6 -----
Fold ROC-AUC: 0.875

----- Fold 7 -----
Fold ROC-AUC: 0.85

----- Fold 8 -----
Fold ROC-AUC: 0.75

----- Fold 9 -----
Fold ROC-AUC: 0.875

----- Fold 10 -----
Fold ROC-AUC: 0.8222

--------------------------------------------------

CatBoost

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # feature scaling
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # handle class imbalance
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)

    class_weights = [
        1,
        neg / pos
    ]

    print("Class Weights:", class_weights)

    # phase-1 optimized multimodal catboost
    model = CatBoostClassifier(

        # core
        loss_function="Logloss",
        eval_metric="AUC",

        # moderate boosting
        iterations=450,

        # simpler trees
        depth=4,

        # slower learning
        learning_rate=0.018,

        # stronger regularization
        l2_leaf_reg=8,

        # lower randomness
        random_strength=1,

        # conservative sampling
        bootstrap_type="Bernoulli",
        subsample=0.75,

        # handle imbalance
        class_weights=class_weights,

        # stable growth
        grow_policy="SymmetricTree",

        random_seed=42,

        verbose=0
    )

    # train model
    model.fit(
        X_train,
        y_train
    )

    # predict probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # threshold
    THRESHOLD = 0.36

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)
Class Weights: [1, np.float64(2.393939393939394)]

Predicted Distribution:
[13 16]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.60      0.73        20
           1       0.50      0.89      0.64         9

    accuracy                           0.69        29
   macro avg       0.71      0.74      0.68        29
weighted avg       0.79      0.69    

CatBoost 5 folds

In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cross validation
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # handle class imbalance
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        class_weights = [
            1,
            neg / pos
        ]

        # phase-1 optimized multimodal catboost
        model = CatBoostClassifier(

            # core
            loss_function="Logloss",
            eval_metric="AUC",

            # moderate boosting
            iterations=450,

            # simpler trees
            depth=4,

            # slower learning
            learning_rate=0.018,

            # stronger regularization
            l2_leaf_reg=8,

            # lower randomness
            random_strength=1,

            # conservative sampling
            bootstrap_type="Bernoulli",
            subsample=0.75,

            # handle imbalance
            class_weights=class_weights,

            # stable growth
            grow_policy="SymmetricTree",

            random_seed=42,

            verbose=0
        )

        # train model
        model.fit(
            X_train,
            y_train
        )

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.36

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.5611

----- Fold 2 -----
Fold ROC-AUC: 0.8938

----- Fold 3 -----
Fold ROC-AUC: 0.8625

----- Fold 4 -----
Fold ROC-AUC: 0.9062

----- Fold 5 -----
Fold ROC-AUC: 0.8655

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.5611 0.8938 0.8625 0.9062 0.8655]

Mean Fold ROC-AUC:
0.8178

Overa

CatBoost 10 folds

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# load dataset
df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important multimodal features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cross validation
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # handle class imbalance
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        class_weights = [
            1,
            neg / pos
        ]

        # phase-1 optimized multimodal catboost
        model = CatBoostClassifier(

            # core
            loss_function="Logloss",
            eval_metric="AUC",

            # moderate boosting
            iterations=450,

            # simpler trees
            depth=4,

            # slower learning
            learning_rate=0.018,

            # stronger regularization
            l2_leaf_reg=8,

            # lower randomness
            random_strength=1,

            # conservative sampling
            bootstrap_type="Bernoulli",
            subsample=0.75,

            # handle imbalance
            class_weights=class_weights,

            # stable growth
            grow_policy="SymmetricTree",

            random_seed=42,

            verbose=0
        )

        # train model
        model.fit(
            X_train,
            y_train
        )

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.36

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.42

----- Fold 2 -----
Fold ROC-AUC: 0.65

----- Fold 3 -----
Fold ROC-AUC: 0.825

----- Fold 4 -----
Fold ROC-AUC: 1.0

----- Fold 5 -----
Fold ROC-AUC: 0.925

----- Fold 6 -----
Fold ROC-AUC: 0.825

----- Fold 7 -----
Fold ROC-AUC: 0.925

----- Fold 8 -----
Fold ROC-AUC: 0.85

----- Fold 9 -----
Fold ROC-AUC: 0.85

----- Fold 10 -----
Fold ROC-AUC: 0.8667

-------------------------------------------------

FNN

In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# gpu setup

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("-" * 60)
print("device info")
print("-" * 60)

print("Using Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# load dataset

df = pd.read_csv("../../data/multimodal_audio_text_features.csv")

df["phase"] = (
    df["phase"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("\nDataset Shape:", df.shape)

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns

remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# custom dataset

class DepressionDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

# phase-1 optimized dnn

class DepressionDNN(nn.Module):

    def __init__(self, input_dim):

        super(DepressionDNN, self).__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.45),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.40),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.35),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        return self.network(x).squeeze()

# train function

def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"training for {phase_name}")
    print("-" * 60)

    df_phase = df_phase.dropna()

    # features & labels

    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)

    print("Class Distribution:", np.bincount(y))

    # remove low variance features

    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important features

    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(260, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # feature scaling

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)

    X_test = scaler.transform(X_test)

    # datasets

    train_dataset = DepressionDataset(
        X_train,
        y_train
    )

    test_dataset = DepressionDataset(
        X_test,
        y_test
    )

    # dataloaders

    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=8,
        shuffle=False
    )

    # model

    model = DepressionDNN(
        input_dim=X_train.shape[1]
    ).to(device)

    # class weighting

    class_counts = np.bincount(y_train)

    pos_weight = torch.tensor(
        class_counts[0] / class_counts[1],
        dtype=torch.float32
    ).to(device)

    # loss function

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    # optimizer

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.00015,
        weight_decay=1e-3
    )

    # learning rate scheduler

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=70
    )

    # training

    EPOCHS = 70

    best_auc = 0

    best_probs = None

    print("\nStarting Training...\n")

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)

            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)

            loss = criterion(
                outputs,
                y_batch
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item()

        scheduler.step()

        avg_loss = total_loss / len(train_loader)

        # validation

        model.eval()

        val_probs = []

        with torch.no_grad():

            for X_batch, _ in test_loader:

                X_batch = X_batch.to(device)

                outputs = model(X_batch)

                probs = torch.sigmoid(outputs)

                probs = probs.cpu().numpy()

                val_probs.extend(probs)

        val_auc = roc_auc_score(
            y_test,
            val_probs
        )

        if val_auc > best_auc:

            best_auc = val_auc

            best_probs = val_probs.copy()

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {avg_loss:.4f} "
            f"AUC: {val_auc:.4f}"
        )

    # final evaluation

    final_probs = np.array(best_probs)

    y_pred = (final_probs > 0.36).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    final_auc = roc_auc_score(
        y_test,
        final_probs
    )

    print("Best ROC-AUC:", round(final_auc, 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(
        y_test,
        y_pred
    ))

    return final_auc

# phase-wise data

df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models

roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final results

print("\n" + "-" * 60)
print("final roc-auc scores")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
device info
------------------------------------------------------------
Using Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

------------------------------------------------------------
training for PHASE 1
------------------------------------------------------------
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 260)

Starting Training...

Epoch [1/70] Loss: 1.0013 AUC: 0.6333
Epoch [2/70] Loss: 0.9610 AUC: 0.6611
Epoch [3/70] Loss: 0.9225 AUC: 0.7000
Epoch [4/70] Loss: 1.0111 AUC: 0.6667
Epoch [5/70] Loss: 0.9745 AUC: 0.6944
Epoch [6/70] Loss: 0.8663 AUC: 0.6944
Epoch [7/70] Loss: 0.9236 AUC: 0.7278
Epoch [8/70] Loss: 0.9267 AUC: 0.7333
Epoch [9/70] Loss: 0.8921 AUC: 0.7611
Epoch [10/70] Loss: 0.8986 AU